# Stage B: lpsr-lacd super-resolution fine-tuning

This notebook stages the competition LR/HR pairs, fine-tunes lpsr-lacd in Colab, and exports restored validation images for Stage C.

Notes:
- Stage B uses the paired-frame split from Prompt 1.
- `LOAD_PRE_TRAINED_OCR` can point to the fine-tuned GPLPR checkpoint from Stage A.
- The separate `loss_sr.args.load` OCR path is left `null` by default because the upstream SR loss expects a different OCR artifact.
- Replace the placeholder repo URL and Stage A checkpoint path before running the cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/lpsrocr'
DATASET_ROOT = '/content/drive/MyDrive/icpr2026_lpr/train'
LPSRLACD_REPO = '/content/lpsr-lacd'
SPLIT_DIR = f'{PROJECT_ROOT}/manifests/splits/scenario_b_dev_seed42_n400_v20'
STAGE_DIR = f'{PROJECT_ROOT}/external_data/lpsrlacd_stage'
OUTPUT_DIR = '/content/drive/MyDrive/lpsrocr/outputs/stage_b'
RUN_TAG = 'stage_b_scenario_b_dev_seed42_n400_v20'
STAGE_A_OCR_CKPT = '/content/drive/MyDrive/lpsrocr/outputs/stage_a/checkpoints/<stage_a_run_tag>/best_model.pth'
CHECKPOINT = f'{OUTPUT_DIR}/checkpoints/{RUN_TAG}/best_model.pth'


In [ ]:
!cd /content && git clone <YOUR_PROJECT_REPO_URL> lpsrocr || true
!cd /content && git clone https://github.com/valfride/lpsr-lacd.git lpsr-lacd || true
%cd /content/lpsrocr
!pip install -r requirements.txt
!pip install tensorboardX albumentations opencv-python-headless scikit-image


In [ ]:
# Stage the paired competition data for lpsr-lacd.
!python scripts/stage_b_prepare_lpsrlacd.py \
  --project-root {PROJECT_ROOT} \
  --dataset-root {DATASET_ROOT} \
  --lpsrlacd-repo {LPSRLACD_REPO} \
  --split-dir {SPLIT_DIR} \
  --stage-dir {STAGE_DIR} \
  --output-dir {OUTPUT_DIR} \
  --mode symlink


In [ ]:
# Fine-tune lpsr-lacd.
# Add --resume-sr if you want to resume a previous SR run.
!python scripts/stage_b_train_lpsrlacd.py \
  --project-root {PROJECT_ROOT} \
  --dataset-root {DATASET_ROOT} \
  --lpsrlacd-repo {LPSRLACD_REPO} \
  --split-dir {SPLIT_DIR} \
  --stage-dir {STAGE_DIR} \
  --output-dir {OUTPUT_DIR} \
  --resume-ocr {STAGE_A_OCR_CKPT}


In [ ]:
# Export restored validation images into the Stage C-friendly layout.
!python scripts/stage_b_infer_lpsrlacd.py \
  --project-root {PROJECT_ROOT} \
  --dataset-root {DATASET_ROOT} \
  --lpsrlacd-repo {LPSRLACD_REPO} \
  --split-dir {SPLIT_DIR} \
  --stage-dir {STAGE_DIR} \
  --output-dir {OUTPUT_DIR} \
  --checkpoint {CHECKPOINT}
